# SQL Schema → ORM Generator

Paste a SQL schema and generate equivalent ORM model code across multiple frameworks.

The pipeline:
1. **Input** — Paste raw SQL `CREATE TABLE` statements
2. **LLM Code Generation** — A frontier model converts the schema into idiomatic ORM code
3. **Multi-target output** — Choose from SQLAlchemy, Django ORM, Prisma, or TypeORM
4. **Multi-model support** — Compare outputs from GPT, Claude, Gemini, and Grok
5. **Gradio UI** — Interactive side-by-side interface

**Week 4 concepts used:** Multi-model code generation (Day 3) · Inference providers with OpenAI wrapper (Day 4) · Interactive Gradio UI (Day 5)

In [ ]:
%pip install -q openai gradio python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")

# Fall back to Google Colab secrets if not found in .env
if not openai_api_key or not anthropic_api_key or not google_api_key:
    try:
        from google.colab import userdata
        openai_api_key = openai_api_key or userdata.get("OPENAI_API_KEY")
        anthropic_api_key = anthropic_api_key or userdata.get("ANTHROPIC_API_KEY")
        google_api_key = google_api_key or userdata.get("GOOGLE_API_KEY")
        grok_api_key = grok_api_key or userdata.get("GROK_API_KEY")
    except Exception:
        pass

# Initialize clients using OpenAI wrapper pattern (Week 4 Day 4)
openai_client = OpenAI(api_key=openai_api_key)
anthropic_client = OpenAI(
    api_key=anthropic_api_key,
    base_url="https://api.anthropic.com/v1/",
)
gemini_client = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
grok_client = OpenAI(
    api_key=grok_api_key,
    base_url="https://api.x.ai/v1",
)

# Model registry
models = ["gpt-4o-mini", "claude-sonnet-4-5-20250929", "gemini-2.5-flash", "grok-3-mini"]

clients = {
    "gpt-4o-mini": openai_client,
    "claude-sonnet-4-5-20250929": anthropic_client,
    "gemini-2.5-flash": gemini_client,
    "grok-3-mini": grok_client,
}

for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROK_API_KEY"]:
    val = os.getenv(key) or locals().get(key.lower().replace("_api_key", "_api_key"))
    status = f"begins {val[:8]}" if val else "NOT SET"
    print(f"{key}: {status}")

## How It Works

SQL schemas define database structure, but every framework has its own way of
representing models in code. Manually translating `CREATE TABLE` statements into
ORM models is tedious and error-prone — especially when you need to handle
foreign keys, indexes, and constraints across different frameworks.

This tool uses frontier LLMs to:
- Parse raw SQL and understand table relationships
- Generate idiomatic ORM code with proper type mappings, relationships, and constraints
- Support 4 popular ORM frameworks across Python and TypeScript

Each ORM target has a tailored system prompt that guides the model to produce
framework-specific patterns (e.g., `relationship()` in SQLAlchemy, `ForeignKey` in Django,
`@relation` in Prisma, `@ManyToOne` in TypeORM).

In [ ]:
# ORM target configurations
ORM_TARGETS = {
    "SQLAlchemy": {
        "language": "python",
        "system_prompt": """You are an expert Python developer specializing in SQLAlchemy.
Convert the provided SQL schema into SQLAlchemy ORM models using the declarative base pattern.

Requirements:
- Use `DeclarativeBase` (SQLAlchemy 2.0+ style)
- Use `Mapped` and `mapped_column` for type-annotated columns
- Include `relationship()` for foreign keys with appropriate `back_populates`
- Map SQL types to SQLAlchemy types (VARCHAR → String, SERIAL → Integer with autoincrement, etc.)
- Preserve constraints: UNIQUE, NOT NULL, DEFAULT values
- Include necessary imports at the top

Respond ONLY with Python code. No explanations.""",
    },
    "Django ORM": {
        "language": "python",
        "system_prompt": """You are an expert Python developer specializing in Django.
Convert the provided SQL schema into Django ORM models.

Requirements:
- Each table becomes a `models.Model` subclass
- Use appropriate Django field types (CharField, TextField, BooleanField, DateTimeField, etc.)
- Use `models.ForeignKey` with `on_delete=models.CASCADE` and `related_name`
- Set `max_length` for CharField, `unique=True`, `null=True`/`blank=True` as needed
- Use `auto_now_add=True` for created_at timestamps
- Include `class Meta` with `db_table` matching the original table name
- Add `__str__` methods

Respond ONLY with Python code. No explanations.""",
    },
    "Prisma": {
        "language": "typescript",
        "system_prompt": """You are an expert TypeScript developer specializing in Prisma ORM.
Convert the provided SQL schema into a Prisma schema file.

Requirements:
- Use proper Prisma schema syntax (model blocks, field types, attributes)
- Map SQL types to Prisma types (VARCHAR → String, SERIAL → Int with @id @default(autoincrement()), etc.)
- Use `@relation` for foreign keys with proper references
- Include `@unique`, `@default`, `@map` attributes as needed
- Add a `datasource` block for PostgreSQL and a `generator` block for prisma-client-js
- Use `@@map` to preserve original table names if model names differ

Respond ONLY with the Prisma schema. No explanations.""",
    },
    "TypeORM": {
        "language": "typescript",
        "system_prompt": """You are an expert TypeScript developer specializing in TypeORM.
Convert the provided SQL schema into TypeORM entity classes.

Requirements:
- Use TypeORM decorators: @Entity, @PrimaryGeneratedColumn, @Column, @ManyToOne, @OneToMany, @JoinColumn
- Map SQL types to TypeScript types with proper column options
- Set nullable, unique, default, and length options on @Column
- Use @CreateDateColumn for created_at timestamps
- Include proper imports from 'typeorm'
- Add inverse relations with @OneToMany where appropriate

Respond ONLY with TypeScript code. No explanations.""",
    },
}


def generate_orm(sql, model_name, orm_target):
    """Generate ORM code from SQL schema using the selected model and target."""
    if not sql.strip():
        return "Please paste a SQL schema first."

    target_config = ORM_TARGETS[orm_target]
    client = clients[model_name]

    user_prompt = f"""Convert this SQL schema into {orm_target} models:

```sql
{sql}
```"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": target_config["system_prompt"]},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=2000,
    )

    reply = response.choices[0].message.content
    # Strip markdown code fences (Week 4 pattern)
    reply = reply.replace("```python", "").replace("```typescript", "")
    reply = reply.replace("```prisma", "").replace("```sql", "")
    reply = reply.replace("```", "").strip()
    return reply


def generate_all(sql, model_name):
    """Generate ORM code for all 4 targets at once."""
    results = []
    for target in ORM_TARGETS:
        try:
            results.append(generate_orm(sql, model_name, target))
        except Exception as e:
            results.append(f"Error: {e}")
    return results


print("Generation functions ready")
print(f"Available models: {models}")
print(f"Available ORM targets: {list(ORM_TARGETS.keys())}")

In [ ]:
SAMPLE_SQL = """CREATE TABLE users (
    id SERIAL PRIMARY KEY,
    email VARCHAR(255) UNIQUE NOT NULL,
    name VARCHAR(100) NOT NULL,
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE TABLE posts (
    id SERIAL PRIMARY KEY,
    title VARCHAR(255) NOT NULL,
    body TEXT,
    user_id INTEGER REFERENCES users(id),
    published BOOLEAN DEFAULT FALSE,
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE TABLE comments (
    id SERIAL PRIMARY KEY,
    body TEXT NOT NULL,
    post_id INTEGER REFERENCES posts(id),
    user_id INTEGER REFERENCES users(id),
    created_at TIMESTAMP DEFAULT NOW()
);"""

with gr.Blocks(title="SQL → ORM Generator") as ui:
    gr.Markdown("# SQL Schema → ORM Generator")
    gr.Markdown(
        "Paste a SQL schema and generate ORM model code for "
        "**SQLAlchemy**, **Django**, **Prisma**, or **TypeORM**. "
        "Compare outputs across frontier LLMs."
    )

    with gr.Row():
        model = gr.Dropdown(models, label="Model", value=models[0])
        orm_target = gr.Dropdown(
            list(ORM_TARGETS.keys()), label="ORM Target", value="SQLAlchemy"
        )
        convert_btn = gr.Button("Generate", variant="primary")
        convert_all_btn = gr.Button("Generate All 4")

    with gr.Row(equal_height=True):
        sql_input = gr.Code(
            label="SQL Schema",
            language="sql",
            value=SAMPLE_SQL,
            lines=20,
        )
        orm_output = gr.Code(
            label="Generated ORM Code",
            language="python",
            lines=20,
        )

    gr.Markdown("### All Targets (Generate All 4)")

    with gr.Row(equal_height=True):
        sa_output = gr.Code(label="SQLAlchemy", language="python", lines=15)
        dj_output = gr.Code(label="Django ORM", language="python", lines=15)

    with gr.Row(equal_height=True):
        pr_output = gr.Code(label="Prisma", language="typescript", lines=15)
        to_output = gr.Code(label="TypeORM", language="typescript", lines=15)

    convert_btn.click(
        fn=generate_orm,
        inputs=[sql_input, model, orm_target],
        outputs=[orm_output],
    )
    convert_all_btn.click(
        fn=generate_all,
        inputs=[sql_input, model],
        outputs=[sa_output, dj_output, pr_output, to_output],
    )

ui.launch(inbrowser=True)